# MessagesPlaceholder — Injecting Dynamic Chat History into a Template

## The problem this solves

Imagine you're building a chatbot and you have a fixed `ChatPromptTemplate`:

```python
template = ChatPromptTemplate([
    ("system", "You are a helpful customer support agent"),
    ("human", "{query}")
])
```

This is good, but it has no memory — it doesn't know what the user said before.

You COULD add messages manually, but then the template becomes rigid and you can't reuse it.

**`MessagesPlaceholder` is the solution** — it creates a named "slot" in the template where you can inject any list of past messages at runtime.

```python
template = ChatPromptTemplate([
    ("system", "You are a helpful customer support agent"),
    MessagesPlaceholder(variable_name="chat_history"),  # ← the slot
    ("human", "{query}")
])

# Later, fill in the slot with the actual history:
template.invoke({"chat_history": past_messages, "query": "What is my order status?"})
```

## What you'll learn in this notebook

- How `MessagesPlaceholder` creates a dynamic slot in a chat template
- How to load a chat history and inject it into the template
- How the final composed prompt looks
- How this pattern is used in production chatbots

## Prerequisites

- No API key or Ollama needed (this notebook only builds prompts, it doesn't call a model)
- Virtual environment activated

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

/Users/rahuljauhari/Desktop/GenAI - Learning/Langchain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
chat_template = ChatPromptTemplate(
    [
        ("system","You are a helpful customer support agent"),
        MessagesPlaceholder(variable_name="chat_history"), 
        # We can store the intermediate messages in a separate file named chat_history.txt
        # Then in message placeholder we can load them from the file and the LLM will then 
        # have context on what was the previous chats...
        ("human","{query}")
    ]
)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# MessagesPlaceholder expects a list of proper message objects (HumanMessage, AIMessage),
# NOT plain text strings. Here we create a sample conversation history manually.
# In a real chatbot, this list grows as the user and AI exchange messages.

chat_history = [
    HumanMessage(content="Hi! My name is Alice."),
    AIMessage(content="Hello Alice! How can I help you today?"),
    HumanMessage(content="I need help with my order."),
    AIMessage(content="Of course! Could you please provide your order number?"),
]

print(f"Loaded {len(chat_history)} messages into chat history")

In [ ]:
result = chat_template.invoke({"query": "What is my order status?", "chat_history": chat_history})
result

## What the output shows

Notice that the `chat_history` messages appear **between** the system message and the human query, exactly as intended. The model will see:

1. `SystemMessage` — "You are a helpful customer support agent"
2. `HumanMessage` — "Hi! My name is Alice."
3. `AIMessage` — "Hello Alice! How can I help you today?"
4. `HumanMessage` — "I need help with my order."
5. `AIMessage` — "Of course! Could you please provide your order number?"
6. `HumanMessage` — "What is my order status?"  ← the new query

This gives the model full context of who Alice is and what she needs.

## Key takeaway

- `MessagesPlaceholder` expects **proper message objects** (`HumanMessage`, `AIMessage`), not plain text strings
- In a real chatbot app, you maintain `chat_history` as a list that grows with each turn
- The template is reusable — you fill in `chat_history` and `query` differently every call